# Phase 5 — Exploratory Data Analysis (EDA)
## Parkinson's Disease Prediction Using Typing Characteristics

**Primary Dataset:** Tappy Extended Keystroke Dataset, Mendeley Data Version 3 (DOI: 10.17632/z39mhdsynx.3)  
**Working Branch:** `praneeth`  
**Scope & Phase Boundary:** Strict Phase 5 exploratory analysis. No model fitting, preprocessing fitting, scaling, clipping, imputation, or synthetic resampling is executed here.  
**Test Set Quarantine:** All distributions, descriptive statistics, outlier screenings, correlation heatmaps, and target comparisons strictly isolate the prospective holdout test set. Only training participants are explored.
**Fail-Closed Gate Contract:** Execution is governed by the Phase 4 Readiness Gate. If the canonical Phase 6 feature table is missing, truncated, or vitiated, execution halts immediately with a diagnostic error.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure paths relative to repository root
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.data_pipeline.eda_engine import (
    evaluate_readiness_gate,
    enforce_readiness_gate,
    build_feature_inventory,
    compute_missingness_summary,
    compute_descriptive_statistics,
    screen_outliers_and_anomalies,
    analyze_feature_correlations,
    compute_training_class_summary,
    generate_eda_figures,
)

print(f'EDA environment configured. Project root: {repo_root}')

## 1. Phase 4 Readiness Gate Evaluation
Before performing exploratory analysis on typing features, we must verify the 8 mandatory Phase 4 readiness gate criteria:
1. Canonical feature table exists and is not the rejected legacy candidate.
2. Split files exist at authoritative paths.
3. Exactly one row per participant with unique IDs.
4. Complete and valid binary target labels consistent with authoritative cohort reconciliation.
5. Participant coverage matches verified eligible cohort ($N=597$).
6. Split partition integrity confirmed (477 training / 120 test, disjoint, exact union).
7. Feature schema verified (typing predictors only; demographic and clinical covariates excluded).
8. Predictor values unvitiated (zero date corruption $>100,000$, zero infinite values).

**Fail-Closed Rule:** If any check fails, `enforce_readiness_gate` halts execution immediately.

In [ ]:
# Canonical feature table path
canonical_features_path = repo_root / 'data' / 'phase6_recovery' / 'features' / 'participant_features_raw.parquet'

# Authoritative split paths
train_ids_path = repo_root / 'data' / 'phase2_recovery' / 'splits' / 'train_ids.csv'
if not train_ids_path.is_file():
    train_ids_path = repo_root / 'data' / 'processed' / 'splits' / 'train_ids.csv'

test_ids_path = repo_root / 'data' / 'phase2_recovery' / 'splits' / 'test_ids.csv'
if not test_ids_path.is_file():
    test_ids_path = repo_root / 'data' / 'processed' / 'splits' / 'test_ids.csv'

gate_results = evaluate_readiness_gate(
    features_path=canonical_features_path,
    train_ids_path=train_ids_path,
    test_ids_path=test_ids_path,
    eligible_count=597,
    expected_train_count=477,
    expected_test_count=120,
    expected_feature_count=153,
    allow_legacy_fallback=False,
)

print('Readiness Gate Evaluation Results:')
print(f'  Overall Verdict: {gate_results["verdict"]}')
for check_name, check_val in gate_results['checks'].items():
    status_sym = '[PASS]' if check_val else '[FAIL]'
    print(f'  {status_sym} {check_name}: {check_val}')

if gate_results['blockers']:
    print('\nIdentified Blockers:')
    for b in gate_results['blockers']:
        print(f'  - {b}')

# Enforce fail-closed gate: halts execution if prerequisites are unmet
enforce_readiness_gate(gate_results)
print('\nPhase 4 Readiness Gate PASSED: proceeding with training-only exploratory data analysis.')

## 2. Cohort Loading & Training-Only Partitioning
We load the verified feature table and partition the cohort strictly using the authoritative split manifests.
The held-out test partition is strictly quarantined to prevent data leakage.

In [ ]:
if canonical_features_path.suffix.lower() == '.parquet':
    df_all = pd.read_parquet(canonical_features_path)
else:
    df_all = pd.read_csv(canonical_features_path)

train_df = pd.read_csv(train_ids_path)
test_df = pd.read_csv(test_ids_path)
train_ids = set(train_df['participant_id'])
test_ids = set(test_df['participant_id'])

# Strict isolation: filter to training participants only
df_train = df_all[df_all['participant_id'].isin(train_ids)].copy().reset_index(drop=True)
df_test = df_all[df_all['participant_id'].isin(test_ids)].copy().reset_index(drop=True)

predictor_cols = [c for c in df_train.columns if c not in ['participant_id', 'parkinsons_label']]

n_pd_train = int((df_train['parkinsons_label'] == 1).sum())
n_ctrl_train = int((df_train['parkinsons_label'] == 0).sum())

print(f'Total cohort shape: {df_all.shape}')
print(f'Training partition: N={len(df_train)} (PD={n_pd_train}, Control={n_ctrl_train}, Prevalence={round(n_pd_train/len(df_train)*100, 2)}%)')
print(f'Held-out test partition (QUARANTINED): N={len(df_test)}')
print(f'Active predictor columns: {len(predictor_cols)}')

## 3. Feature Inventory & Taxonomy
We catalog the typing predictors across their physiological and temporal families:
- Key Hold Time statistics (`hold_*`)
- Latency / Inter-key interval statistics (`latency_*`)
- Flight Time statistics (`flight_*`)
- Hand Dynamics proportions (`hand_*` / `prop_*`)
- Directional and Transition statistics (`*_LL`, `*_RR`, `*_LR`, etc.)

In [ ]:
inv_df = build_feature_inventory(df_train)
print('Feature taxonomy breakdown across predictor families:')
print(inv_df['family'].value_counts())
print('\nIntegrity status distribution:')
print(inv_df['integrity_status'].value_counts())
print('\nSample Feature Inventory Records:')
print(inv_df[['feature_name', 'family', 'unit', 'missing_count', 'missing_pct', 'integrity_status']].head(12).to_string(index=False))

## 4. Dynamic Missingness Analysis
We evaluate missing values across all predictor columns dynamically within the training cohort.

In [ ]:
miss_df = compute_missingness_summary(df_train)
non_zero_miss = miss_df[miss_df['missing_count'] > 0]
print(f'Features with missing values: {len(non_zero_miss)} out of {len(miss_df)}')
if not non_zero_miss.empty:
    print('Top missing features in training cohort:')
    print(non_zero_miss.head(10).to_string(index=False))
else:
    print('Zero missing values observed across all evaluated training predictors.')

## 5. Descriptive Statistics of Training Predictors
We compute univariate descriptive metrics (mean, std, median, IQR, min, max, skewness, kurtosis, and Tukey outlier bounds) dynamically for all training features.

In [ ]:
desc_df = compute_descriptive_statistics(df_train)
key_candidates = ['hold_mean', 'hold_std', 'hold_median', 'hold_cv', 'latency_mean', 'latency_std', 'flight_mean']
available_key = [f for f in key_candidates if f in desc_df['feature_name'].values]
if available_key:
    key_desc = desc_df[desc_df['feature_name'].isin(available_key)][['feature_name', 'mean', 'std', 'median', 'iqr', 'skewness', 'min', 'max']]
    print('Summary of key timing statistics:')
    print(key_desc.to_string(index=False))
else:
    print('Descriptive statistics summary (first 8 features):')
    print(desc_df[['feature_name', 'mean', 'std', 'median', 'iqr', 'skewness', 'min', 'max']].head(8).to_string(index=False))

## 6. Anomaly Screening & Semantic Integrity Audit
We screen training features dynamically for anomalies:
1. Date corruption (> 100,000)
2. Rollover preservation (negative flight times)
3. Statistical Tukey outliers

In [ ]:
outlier_df = screen_outliers_and_anomalies(df_train)
if not outlier_df.empty:
    print('Anomaly Screening Classification breakdown:')
    print(outlier_df['anomaly_classification'].value_counts())
    print('\nSeverity Level breakdown:')
    print(outlier_df['severity_level'].value_counts())
    fatal = outlier_df[outlier_df['severity_level'] == 'FATAL_BLOCKER']
    print(f'\nFatal Blocker Anomalies: {len(fatal)}')
else:
    print('No severe or fatal anomalies flagged in training predictors.')

## 7. Feature Correlation & Collinearity Analysis
We compute pairwise Spearman rank correlations across uncorrupted training features dynamically.
Pairs with $|\rho| \ge 0.90$ are flagged as redundant candidate statistics.

In [ ]:
corr_df = analyze_feature_correlations(df_train, threshold=0.90)
print(f'Total collinear pairs (|rho| >= 0.90): {len(corr_df)}')
if not corr_df.empty:
    print('Top correlated pairs:')
    print(corr_df.head(10).to_string(index=False))

## 8. Exploratory Class Comparisons (Training Set Only)
We compare distributions between Parkinson's and Control participants on representative training features.

In [ ]:
class_df = compute_training_class_summary(df_train)
if not class_df.empty:
    print('Training set class comparison summary:')
    print(class_df[['feature_name', 'control_count', 'control_median', 'pd_count', 'pd_median', 'median_diff_pd_minus_ctrl']].to_string(index=False))

## 9. Diagnostic Figure Generation
We export dynamic diagnostic figures to `reports/phase5_recovery/eda/figures/`.

In [ ]:
fig_dir = repo_root / 'reports' / 'phase5_recovery' / 'eda' / 'figures'
saved_figs = generate_eda_figures(df_train, fig_dir)
print(f'Generated {len(saved_figs)} diagnostic EDA figures under {fig_dir}:')
for p in saved_figs:
    print(f'  - {Path(p).name}')